In [1]:
import os

folder_path = 'benchmarks/results' 
files = os.listdir(folder_path)
print(files)

['20260318_145955', '20260314_195539', '20260316_020327', '20260318_164756', '20260312_103401']


In [2]:
import pandas as pd
import glob

# csv_files = glob.glob(os.path.join(folder_path, '**', '*.csv'), recursive=True)
# print("CSV files found:", csv_files)

csv_files = ['benchmarks/results/20260318_145955/summary.csv', 'benchmarks/results/20260318_164756/summary.csv']

combined_df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)

In [3]:
combined_df = combined_df.drop(columns=['batch_size', 'block_size', 'decode_mode', 'gen_length', 'model_path', 'offload_dir',
                                        'prompt_mode', 'raw_stdout', 'repetitions', 'target_context_len', 'threshold',
                                        'trace_dir', 'use_torch_profiler', 'warmup'])
combined_df

,do_compile,max_mem_alloc_mb,max_mem_reserved_mb,model_key,per_token_time_ms,prompt_len,run_type,tokens_per_s,total_time_s
0,True,3942.153320,4530.0,qwen,48.107190,128,baseline,20.786914,6.157720
1,True,3942.153320,4530.0,qwen,60.924656,128,profiled,16.413716,7.798356
2,True,3971.157715,4582.0,qwen,49.770664,256,baseline,20.092157,6.370645
3,True,3971.157715,4582.0,qwen,61.493253,256,profiled,16.261947,7.871136
4,True,4031.433594,4526.0,qwen,53.516373,512,baseline,18.685870,6.850096
5,True,4031.433594,4526.0,qwen,62.249069,512,profiled,16.064497,7.967881
6,True,4145.182617,4472.0,qwen,59.833819,1024,baseline,16.712956,7.658729
7,True,4145.182617,4472.0,qwen,64.010170,1024,profiled,15.622518,8.193302
8,True,4377.215820,4472.0,qwen,73.723249,2048,baseline,13.564242,9.436576
9,True,4377.215820,4472.0,qwen,75.203079,2048,profiled,13.297328,9.625994


In [4]:
raw_df = combined_df[combined_df['run_type'] == 'baseline']
raw_df.drop(columns=['run_type'])

,do_compile,max_mem_alloc_mb,max_mem_reserved_mb,model_key,per_token_time_ms,prompt_len,tokens_per_s,total_time_s
0,True,3942.153320,4530.0,qwen,48.107190,128,20.786914,6.157720
2,True,3971.157715,4582.0,qwen,49.770664,256,20.092157,6.370645
4,True,4031.433594,4526.0,qwen,53.516373,512,18.685870,6.850096
6,True,4145.182617,4472.0,qwen,59.833819,1024,16.712956,7.658729
8,True,4377.215820,4472.0,qwen,73.723249,2048,13.564242,9.436576
10,True,4841.946289,5036.0,qwen,105.210031,4096,9.504797,13.466884
12,True,3003.943848,3448.0,fast_dllm,24.846691,128,40.246808,3.180376
14,True,3044.843750,3458.0,fast_dllm,19.822984,256,50.446491,2.537342
16,True,3126.791016,3476.0,fast_dllm,36.096145,512,27.703790,4.620307
18,True,3290.685547,3522.0,fast_dllm,44.435082,1024,22.504741,5.687691


In [5]:
baseline_df = combined_df[combined_df['run_type'] == 'baseline'].copy()
index_cols = ['prompt_len']
metric_order = [
    'total_time_s',
    # 'per_token_time_ms',
    'tokens_per_s',
    'max_mem_alloc_mb',
    'max_mem_reserved_mb',
]

models_df_dict = {}
def get_speedup(metric, a, b):
    match metric:
        case 'total_time_s': return a / b
        case _: return b / a 

for model_key in sorted(baseline_df['model_key'].unique()):
    model_df = baseline_df[baseline_df['model_key'] == model_key].copy()

    baseline_table = (
        model_df[model_df['do_compile'] == False]
        .drop(columns=['model_key', 'run_type', 'do_compile'])
        .set_index(index_cols)
        .sort_index()
    )
    compiled_table = (
        model_df[model_df['do_compile'] == True]
        .drop(columns=['model_key', 'run_type', 'do_compile'])
        .set_index(index_cols)
        .sort_index()
    )

    common_index = baseline_table.index.intersection(compiled_table.index)
    baseline_table = baseline_table.loc[common_index]
    compiled_table = compiled_table.loc[common_index]

    metrics = [
        metric for metric in metric_order
        if metric in baseline_table.columns and metric in compiled_table.columns
    ]

    comparison_table = pd.concat(
        {
            metric: pd.DataFrame(
                {
                    'baseline': baseline_table[metric],
                    'compiled': compiled_table[metric],
                    'speedup': get_speedup(metric, baseline_table[metric], compiled_table[metric]),
                }
            )
            for metric in metrics
        },
        axis=1,
    )
    models_df_dict[model_key] = comparison_table


In [6]:
models_df_dict.keys()

dict_keys(['fast_dllm', 'qwen'])

In [7]:
models_df_dict['fast_dllm'].round(2)

total_time_s                  tokens_per_s                   \
               baseline compiled speedup     baseline compiled speedup   
prompt_len                                                               
128                5.10     3.18    1.60        25.08    40.25    1.60   
256                6.43     2.54    2.53        19.90    50.45    2.53   
512                6.71     4.62    1.45        19.08    27.70    1.45   
1024               7.84     5.69    1.38        16.33    22.50    1.38   
2048               9.83     8.09    1.21        13.03    15.82    1.21   
4096              14.88    12.21    1.22         8.60    10.48    1.22   

           max_mem_alloc_mb                  max_mem_reserved_mb           \
                   baseline compiled speedup            baseline compiled   
prompt_len                                                                  
128                 3003.87  3003.94     1.0              3448.0   3448.0   
256                 3044.84  3044.84     1.0              3454.0   3458.0   
512                 3126.79  3126.79     1.0              3458.0   3476.0   
1024                3290.69  3290.69     1.0              3488.0   3522.0   
2048                3624.60  3624.13     1.0              4098.0   4098.0   
4096                4294.86  4292.34     1.0              4636.0   4636.0   

                    
           speedup  
prompt_len          
128           1.00  
256           1.00  
512           1.01  
1024          1.01  
2048          1.00  
4096          1.00

In [8]:
models_df_dict['qwen'].round(2)

total_time_s                  tokens_per_s                   \
               baseline compiled speedup     baseline compiled speedup   
prompt_len                                                               
128               12.79     6.16    2.08        10.01    20.79    2.08   
256               12.20     6.37    1.92        10.49    20.09    1.92   
512               12.34     6.85    1.80        10.37    18.69    1.80   
1024              12.49     7.66    1.63        10.25    16.71    1.63   
2048              13.61     9.44    1.44         9.40    13.56    1.44   
4096              16.83    13.47    1.25         7.61     9.50    1.25   

           max_mem_alloc_mb                  max_mem_reserved_mb           \
                   baseline compiled speedup            baseline compiled   
prompt_len                                                                  
128                 3914.37  3942.15    1.01              4504.0   4530.0   
256                 3929.37  3971.16    1.01              4530.0   4582.0   
512                 3984.84  4031.43    1.01              4530.0   4526.0   
1024                4048.42  4145.18    1.02              4472.0   4472.0   
2048                4212.96  4377.22    1.04              4474.0   4472.0   
4096                4542.03  4841.95    1.07              4616.0   5036.0   

                    
           speedup  
prompt_len          
128           1.01  
256           1.01  
512           1.00  
1024          1.00  
2048          1.00  
4096          1.09